# 🎯 Урок 12 — Твой мини-проект «Предсказатель»

Сегодня ты сам проходишь весь путь ML-проекта. Заполни **6 разделов**. Ниже уже есть рабочий пример на Titanic — используй его как образец и меняй под свой датасет.

> ★ **Обязательно:** baseline, Pipeline и честная метрика (урок 11).

**Оценивается по рубрике (10 баллов):** постановка (2) · ML/Pipeline+baseline (3) · визуализация (2) · интерпретация (3).

## 1. Задача ✍️
Опиши своими словами: что предсказываешь, какие признаки, какая метрика главная.

*Пример:* предсказываю, выжил ли пассажир Titanic; признаки — пол, класс, возраст…; главная метрика — recall.

*Моя задача:* …

## 2. Данные
Загрузи датасет и раздели на train/test. (Можешь заменить Titanic на свой.)

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split

df = sns.load_dataset('titanic')          # 👈 замени на свой датасет при желании
num = ['age','fare','sibsp','parch']       # 👈 свои числовые признаки
cat = ['sex','pclass','embarked']          # 👈 свои категориальные признаки
X = df[num+cat]; y = df['survived']         # 👈 свой target

X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)
df[num+cat].head()

## 3. Baseline
Простая модель, которую надо обыграть.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
base = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print(f'Baseline accuracy: {accuracy_score(y_te, base.predict(X_te)):.0%}')

## 4. Pipeline (обязательно)
Вся подготовка + модель в одном объекте.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat),
])
model = Pipeline([('prep',prep),('forest',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print('Pipeline обучен ✅')

## 5. Оценка
Метрика (урок 11) + сравнение с baseline + кросс-валидация.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
print(f'Модель: {accuracy_score(y_te, model.predict(X_te)):.0%}  |  baseline: {accuracy_score(y_te, base.predict(X_te)):.0%}')
print(classification_report(y_te, model.predict(X_te)))
cv = cross_val_score(model, X, y, cv=5)
print(f'Кросс-валидация: {cv.mean():.1%} ± {cv.std():.1%}')

## 6. Вывод ✍️
Ответь: обыграл ли baseline? какая метрика главная и почему? где модель ошибается? что улучшить?

*Мой вывод:* …

### 📊 Для слайдов (по желанию) — важность признаков

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
# работает, если последний шаг — дерево/лес
forest = model.named_steps['forest']
names = model.named_steps['prep'].get_feature_names_out()
imp = pd.Series(forest.feature_importances_, index=names).sort_values().tail(8)
imp.plot(kind='barh', color='#5B4FC4'); plt.title('Важность признаков'); plt.tight_layout(); plt.show()

---
### 🏁 Чек-лист перед сдачей
- [ ] Задача описана словами
- [ ] Есть baseline
- [ ] Модель собрана как **Pipeline**
- [ ] Есть честная метрика и сравнение с baseline
- [ ] Есть вывод словами
- [ ] Готовы 3 слайда: задача → данные → результат → вывод

🎉 Удачи на защите!